# Belief Rating Prediction – MovieLens Beliefs Dataset
**Đồ án dự đoán expected rating (belief) từ MovieLens Beliefs Dataset**

**Research Question:**  
> *Liệu lịch sử đánh giá của người dùng và thông tin thể loại phim có đủ để dự đoán tốt belief rating (rating kỳ vọng) cho phim chưa xem không? Và độ chắc chắn (userCertainty) của người dùng có tương quan với sai số dự đoán không?*

---
**Outline:**
- Section 0 – Setup & Load Data  
- Section 1 – EDA & Data Visualization  
- Section 2 – Preprocessing & Time-based Split  
- Section 3 – Baselines  
- Section 4 – Biased Matrix Factorization (SGD) – Model Chính  
- Section 5 – Result Analysis & Visualizations  
- Section 6 – Ablation / Leakage Demo  
- Section 7 – Conclusion  


---
## Section 0 – Setup & Load Data

Cài đặt thư viện cần thiết, cấu hình đường dẫn dữ liệu, và nạp các file CSV.  
Nếu file dữ liệu chưa có, notebook sẽ in hướng dẫn thay vì báo lỗi ngay.


In [ ]:
# ── Install dependencies (Colab) ─────────────────────────────────────────
# Bỏ comment dòng dưới nếu chạy trên Google Colab
# !pip install -q pandas numpy scikit-learn matplotlib seaborn tqdm


In [ ]:
# ── Standard imports ────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


### 0.1 – Cấu hình đường dẫn dữ liệu

Chỉnh `DATA_DIR` trỏ tới thư mục chứa các file CSV.  
**Google Drive:** mount drive và đặt ví dụ `DATA_DIR = "/content/drive/MyDrive/movielens_beliefs"`  
**Upload thủ công trên Colab:** `DATA_DIR = "/content"`


In [ ]:
# ── [CONFIGURE ME] ──────────────────────────────────────────────────────
DATA_DIR = "data"   # ← đổi thành đường dẫn thực tế của bạn

# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = "/content/drive/MyDrive/movielens_beliefs"


### 0.2 – Load CSV files

File mong đợi trong `DATA_DIR`:
- `belief_data.csv`  
- `user_rating_history.csv`  
- `movies.csv`  
- `user_recommendation_history.csv` *(optional – dùng trong ablation)*  
- `movie_elicitation_set.csv` *(optional – dùng cho EDA)*


In [ ]:
def load_csv(filename, **kwargs):
    """Load a CSV from DATA_DIR; return None with a guidance message if missing."""
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        print(f"[WARNING] File not found: {path}")
        print(f"          → Please place '{filename}' inside DATA_DIR='{DATA_DIR}'")
        return None
    df = pd.read_csv(path, **kwargs)
    print(f"[OK] Loaded {filename}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df


belief_raw      = load_csv("belief_data.csv")
ratings_raw     = load_csv("user_rating_history.csv")
movies_raw      = load_csv("movies.csv")
rec_history     = load_csv("user_recommendation_history.csv")
elicitation_set = load_csv("movie_elicitation_set.csv")

DATA_AVAILABLE = belief_raw is not None and ratings_raw is not None and movies_raw is not None
if not DATA_AVAILABLE:
    print("\n⚠️  One or more required data files are missing.")
    print("   The notebook will skip computation cells and only display code / explanations.")
else:
    print("\n✅ All required data files loaded successfully.")


In [ ]:
if DATA_AVAILABLE:
    print("=== belief_data sample ===")
    display(belief_raw.head(3))
    print("\n=== user_rating_history sample ===")
    display(ratings_raw.head(3))
    print("\n=== movies sample ===")
    display(movies_raw.head(3))


---
## Section 1 – EDA & Data Visualization

Phân tích khám phá dữ liệu (Exploratory Data Analysis):
- Phân phối `isSeen`, `userPredictRating`, `userCertainty`  
- Top genres phổ biến  
- Xu hướng số lượng beliefs theo `month_idx`  
- Correlation heatmap giữa các trường số


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping EDA section.")
else:
    print("belief_data columns:", belief_raw.columns.tolist())
    print("\nData types:\n", belief_raw.dtypes)
    print("\nNull counts:\n", belief_raw.isnull().sum())


In [ ]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # 1) isSeen distribution
    ax = axes[0]
    vc = belief_raw["isSeen"].value_counts().sort_index()
    labels = {-1: "No response\n(-1)", 0: "Not seen\n(0)", 1: "Seen\n(1)"}
    ax.bar([labels.get(k, str(k)) for k in vc.index], vc.values,
           color=sns.color_palette("muted", len(vc)))
    ax.set_title("Distribution of isSeen", fontsize=13)
    ax.set_ylabel("Count")
    for p, v in zip(ax.patches, vc.values):
        ax.text(p.get_x() + p.get_width()/2, p.get_height() + 20,
                f"{v:,}", ha="center", fontsize=9)

    # 2) userPredictRating distribution (isSeen=0)
    ax = axes[1]
    pred_vals = belief_raw.loc[belief_raw["isSeen"] == 0, "userPredictRating"].dropna()
    ax.hist(pred_vals, bins=20, color=sns.color_palette("muted")[1], edgecolor="white")
    ax.set_title("Distribution of userPredictRating\n(isSeen=0)", fontsize=13)
    ax.set_xlabel("Rating")
    ax.set_ylabel("Count")
    ax.axvline(pred_vals.mean(), color="red", linestyle="--", label=f"Mean={pred_vals.mean():.2f}")
    ax.legend()

    # 3) userCertainty distribution (isSeen=0)
    ax = axes[2]
    cert_vals = belief_raw.loc[belief_raw["isSeen"] == 0, "userCertainty"].dropna()
    vc2 = cert_vals.value_counts().sort_index()
    ax.bar(vc2.index.astype(str), vc2.values, color=sns.color_palette("muted")[2])
    ax.set_title("Distribution of userCertainty\n(isSeen=0)", fontsize=13)
    ax.set_xlabel("Certainty (1–5)")
    ax.set_ylabel("Count")

    plt.tight_layout()
    plt.savefig(os.path.join(DATA_DIR, "eda_basic_distributions.png"), dpi=120, bbox_inches="tight") if DATA_DIR != "data" else None
    plt.show()


In [ ]:
if DATA_AVAILABLE:
    # Top genres from movies_raw
    genres_expanded = (
        movies_raw["genres"]
        .str.split("|")
        .explode()
        .value_counts()
        .head(15)
    )
    plt.figure(figsize=(10, 4))
    sns.barplot(x=genres_expanded.values, y=genres_expanded.index, palette="viridis")
    plt.title("Top 15 Movie Genres (movies.csv)", fontsize=13)
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
if DATA_AVAILABLE and elicitation_set is not None and "month_idx" in elicitation_set.columns:
    monthly = elicitation_set.groupby("month_idx").size().reset_index(name="count")
    plt.figure(figsize=(10, 4))
    plt.plot(monthly["month_idx"], monthly["count"], marker="o", color=sns.color_palette("muted")[0])
    plt.fill_between(monthly["month_idx"], monthly["count"], alpha=0.2)
    plt.title("Number of Elicitations per Month Index", fontsize=13)
    plt.xlabel("month_idx")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()
elif DATA_AVAILABLE:
    print("[INFO] elicitation_set not available or missing 'month_idx' – skipping trend plot.")


In [ ]:
if DATA_AVAILABLE:
    # Correlation heatmap on numeric columns of belief_raw (isSeen=0 subset)
    belief_seen0 = belief_raw[belief_raw["isSeen"] == 0].copy()
    num_cols = belief_seen0.select_dtypes(include=[np.number]).columns.tolist()
    # Keep only meaningful ones if available
    keep = [c for c in ["userPredictRating", "userCertainty", "systemPredictRating"] if c in num_cols]
    if len(keep) >= 2:
        corr = belief_seen0[keep].corr()
        plt.figure(figsize=(6, 4))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                    linewidths=0.5, square=True)
        plt.title("Correlation Heatmap (belief rows, isSeen=0)", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] Not enough numeric columns for heatmap.")


In [ ]:
if DATA_AVAILABLE:
    # Additional: distribution of ratings in user_rating_history
    plt.figure(figsize=(8, 4))
    ratings_raw["rating"].hist(bins=20, color=sns.color_palette("muted")[3], edgecolor="white")
    plt.title("Distribution of Ratings (user_rating_history.csv)", fontsize=13)
    plt.xlabel("Rating")
    plt.ylabel("Count")
    plt.axvline(ratings_raw["rating"].mean(), color="red", linestyle="--",
                label=f"Mean={ratings_raw['rating'].mean():.2f}")
    plt.legend()
    plt.tight_layout()
    plt.show()


---
## Section 2 – Preprocessing & Time-based Split

### 2.1 Tại sao phải dùng time-based split?

Random split inflate metric vì:
- Cùng user-item pair có thể xuất hiện ở cả train và test.
- Thông tin tương lai "rò rỉ" vào train.

**Chiến lược:** per-user chronological split theo `tstamp`.  
- Train: 70% elicitation sớm nhất của mỗi user  
- Validation: 15% tiếp theo  
- Test: 15% mới nhất

### 2.2 Handling missing data (MNAR)

`isSeen = -1` là non-response – **không** phải negative label; loại bỏ khỏi bài toán.  
`userPredictRating` chỉ tồn tại khi `isSeen = 0` → chỉ lấy những hàng đó làm dataset chính.


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping preprocessing section.")
else:
    # ── Step 1: Filter belief rows ─────────────────────────────────────────
    belief = belief_raw[
        (belief_raw["isSeen"] == 0) &
        (belief_raw["userPredictRating"].notna())
    ].copy()
    print(f"Belief rows (isSeen=0, userPredictRating not null): {len(belief):,}")

    # ── Step 2: Merge movie genres ─────────────────────────────────────────
    # One-hot encode genres
    movies = movies_raw[["movieId", "genres"]].copy()
    movies["genres"] = movies["genres"].fillna("(no genres listed)")
    all_genres = set()
    movies["genres"].str.split("|").apply(all_genres.update)
    all_genres.discard("(no genres listed)")
    GENRE_COLS = sorted(all_genres)
    for g in GENRE_COLS:
        movies[f"genre_{g}"] = movies["genres"].str.contains(g, regex=False).astype(int)
    movies.drop(columns=["genres"], inplace=True)

    belief = belief.merge(movies, on="movieId", how="left")

    # ── Step 3: Add user rating stats from history ─────────────────────────
    user_stats = (
        ratings_raw
        .groupby("userId")["rating"]
        .agg(user_mean_rating="mean", user_rating_count="count", user_rating_std="std")
        .reset_index()
    )
    user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0)

    belief = belief.merge(user_stats, on="userId", how="left")

    # Fill NaN for new users with global mean and 0 count
    global_mean = ratings_raw["rating"].mean()
    belief["user_mean_rating"] = belief["user_mean_rating"].fillna(global_mean)
    belief["user_rating_count"] = belief["user_rating_count"].fillna(0)
    belief["user_rating_std"] = belief["user_rating_std"].fillna(0)

    # ── Step 4: Add movie rating stats from history ────────────────────────
    movie_stats = (
        ratings_raw
        .groupby("movieId")["rating"]
        .agg(movie_mean_rating="mean", movie_rating_count="count")
        .reset_index()
    )
    belief = belief.merge(movie_stats, on="movieId", how="left")
    belief["movie_mean_rating"] = belief["movie_mean_rating"].fillna(global_mean)
    belief["movie_rating_count"] = belief["movie_rating_count"].fillna(0)

    print(f"Belief dataset after feature engineering: {belief.shape}")
    display(belief.head(3))


In [ ]:
if DATA_AVAILABLE:
    # ── Step 5: Time-based per-user split ──────────────────────────────────
    def time_based_split(df, tstamp_col="tstamp", train_ratio=0.70, val_ratio=0.15):
        """
        Per-user chronological split.
        Returns train, val, test DataFrames.
        """
        train_rows, val_rows, test_rows = [], [], []
        for user_id, group in df.groupby("userId"):
            group_sorted = group.sort_values(tstamp_col)
            n = len(group_sorted)
            n_train = max(1, int(n * train_ratio))
            n_val   = max(0, int(n * val_ratio))
            train_rows.append(group_sorted.iloc[:n_train])
            val_rows.append(group_sorted.iloc[n_train:n_train + n_val])
            test_rows.append(group_sorted.iloc[n_train + n_val:])
        return (
            pd.concat(train_rows, ignore_index=True),
            pd.concat(val_rows,   ignore_index=True),
            pd.concat(test_rows,  ignore_index=True),
        )

    train_df, val_df, test_df = time_based_split(belief)
    print(f"Train : {len(train_df):,} rows ({len(train_df)/len(belief)*100:.1f}%)")
    print(f"Val   : {len(val_df):,} rows ({len(val_df)/len(belief)*100:.1f}%)")
    print(f"Test  : {len(test_df):,} rows ({len(test_df)/len(belief)*100:.1f}%)")


In [ ]:
# ── Evaluation helpers ────────────────────────────────────────────────────
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

CLIP_MIN, CLIP_MAX = 0.5, 5.0

def clip_preds(preds):
    """Clip predictions to valid rating range [0.5, 5.0]."""
    return np.clip(preds, CLIP_MIN, CLIP_MAX)

def evaluate(name, y_true, y_pred):
    y_pred_clipped = clip_preds(y_pred)
    r = rmse(y_true, y_pred_clipped)
    m = mae(y_true, y_pred_clipped)
    print(f"[{name:40s}]  RMSE={r:.4f}  MAE={m:.4f}")
    return {"model": name, "RMSE": r, "MAE": m}

results = []   # accumulate all results here


---
## Section 3 – Baselines

Chạy các baseline đơn giản để làm mức tham chiếu:

1. **Global Mean**: dự đoán trung bình toàn cục của `userPredictRating` trên tập train.  
2. **User Mean**: dự đoán trung bình của từng user (fallback về global mean).  
3. **Movie Mean**: dự đoán trung bình của từng phim (fallback về global mean).  
4. **Additive Bias (user + movie bias)**: $\hat{r} = \mu + b_u + b_i$.  
5. **Ridge Regression** (trên genre + user/movie stats, không dùng `systemPredictRating`).  
6. **Random Forest** (optional, thêm để so sánh).


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping baselines section.")
else:
    TARGET = "userPredictRating"
    y_train = train_df[TARGET].values
    y_val   = val_df[TARGET].values
    y_test  = test_df[TARGET].values


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 1: Global Mean ────────────────────────────────────────────
    global_mean_pred = y_train.mean()
    pred_gm_val = np.full(len(y_val),  global_mean_pred)
    pred_gm_tst = np.full(len(y_test), global_mean_pred)
    res = evaluate("Global Mean (val)",  y_val,  pred_gm_val)
    results.append({**res, "split": "val"})
    res = evaluate("Global Mean (test)", y_test, pred_gm_tst)
    results.append({**res, "split": "test"})


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 2: User Mean ──────────────────────────────────────────────
    user_mean_map = train_df.groupby("userId")[TARGET].mean().to_dict()

    def predict_user_mean(df):
        return np.array([user_mean_map.get(uid, global_mean_pred) for uid in df["userId"]])

    res = evaluate("User Mean (val)",  y_val,  predict_user_mean(val_df))
    results.append({**res, "split": "val"})
    res = evaluate("User Mean (test)", y_test, predict_user_mean(test_df))
    results.append({**res, "split": "test"})


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 3: Movie Mean ─────────────────────────────────────────────
    movie_mean_map = train_df.groupby("movieId")[TARGET].mean().to_dict()

    def predict_movie_mean(df):
        return np.array([movie_mean_map.get(mid, global_mean_pred) for mid in df["movieId"]])

    res = evaluate("Movie Mean (val)",  y_val,  predict_movie_mean(val_df))
    results.append({**res, "split": "val"})
    res = evaluate("Movie Mean (test)", y_test, predict_movie_mean(test_df))
    results.append({**res, "split": "test"})


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 4: Additive Bias ──────────────────────────────────────────
    # r̂ = μ + b_u + b_i
    # b_u = user_mean - μ,  b_i = movie_mean - μ
    mu = global_mean_pred
    b_u = {uid: (m - mu) for uid, m in user_mean_map.items()}
    b_i = {mid: (m - mu) for mid, m in movie_mean_map.items()}

    def predict_additive_bias(df):
        preds = []
        for _, row in df[["userId", "movieId"]].iterrows():
            p = mu + b_u.get(row["userId"], 0.0) + b_i.get(row["movieId"], 0.0)
            preds.append(p)
        return np.array(preds)

    res = evaluate("Additive Bias (val)",  y_val,  predict_additive_bias(val_df))
    results.append({**res, "split": "val"})
    res = evaluate("Additive Bias (test)", y_test, predict_additive_bias(test_df))
    results.append({**res, "split": "test"})


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 5: Ridge Regression ──────────────────────────────────────
    # Features: genre one-hot + user/movie mean rating + counts
    # NOTE: systemPredictRating is intentionally excluded (anti-leakage)
    FEATURE_COLS = (
        [f"genre_{g}" for g in GENRE_COLS]
        + ["user_mean_rating", "user_rating_count", "user_rating_std",
           "movie_mean_rating", "movie_rating_count"]
    )
    # Keep only cols that exist in the dataframe
    FEATURE_COLS = [c for c in FEATURE_COLS if c in train_df.columns]

    X_train = train_df[FEATURE_COLS].fillna(0).values
    X_val   = val_df[FEATURE_COLS].fillna(0).values
    X_test  = test_df[FEATURE_COLS].fillna(0).values

    ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
    ridge.fit(X_train, y_train)

    res = evaluate("Ridge Regression (val)",  y_val,  ridge.predict(X_val))
    results.append({**res, "split": "val"})
    res = evaluate("Ridge Regression (test)", y_test, ridge.predict(X_test))
    results.append({**res, "split": "test"})


In [ ]:
if DATA_AVAILABLE:
    # ── Baseline 6: Random Forest (optional) ──────────────────────────────
    rf = RandomForestRegressor(n_estimators=100, max_depth=8, n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(X_train, y_train)

    res = evaluate("Random Forest (val)",  y_val,  rf.predict(X_val))
    results.append({**res, "split": "val"})
    res = evaluate("Random Forest (test)", y_test, rf.predict(X_test))
    results.append({**res, "split": "test"})


---
## Section 4 – Biased Matrix Factorization (SGD) – Model Chính

### 4.1 Lý thuyết thuật toán

**Công thức dự đoán:**

$$\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$$

Trong đó:
- $\mu$ – global mean  
- $b_u$ – user bias  
- $b_i$ – item (movie) bias  
- $\mathbf{p}_u \in \mathbb{R}^k$ – user latent factor  
- $\mathbf{q}_i \in \mathbb{R}^k$ – item latent factor  

**Hàm mất mát (Loss function):**

$$\mathcal{L} = \sum_{(u,i) \in \mathcal{D}} (r_{ui} - \hat{r}_{ui})^2 + \lambda \left( \|\mathbf{p}_u\|^2 + \|\mathbf{q}_i\|^2 + b_u^2 + b_i^2 \right)$$

**SGD Update rules** (mini-batch / online):

Với mỗi mẫu $(u, i, r_{ui})$, tính lỗi $e_{ui} = r_{ui} - \hat{r}_{ui}$, rồi cập nhật:

$$b_u \leftarrow b_u + \eta \left( e_{ui} - \lambda b_u \right)$$
$$b_i \leftarrow b_i + \eta \left( e_{ui} - \lambda b_i \right)$$
$$\mathbf{p}_u \leftarrow \mathbf{p}_u + \eta \left( e_{ui} \mathbf{q}_i - \lambda \mathbf{p}_u \right)$$
$$\mathbf{q}_i \leftarrow \mathbf{q}_i + \eta \left( e_{ui} \mathbf{p}_u - \lambda \mathbf{q}_i \right)$$

**Regularization:** ngăn overfitting cho các user/item sparse (ít tương tác).

**Tại sao không dùng `systemPredictRating`?**  
Đây là output của recommender nội bộ của dataset → dùng làm feature sẽ gây *policy leakage* (mô hình chỉ học lại hệ thống cũ, không phản ánh năng lực thực sự). Leakage được demo riêng trong Section 6.


In [ ]:
class BiasedMF:
    """
    Biased Matrix Factorization trained with SGD.

    Parameters
    ----------
    n_factors : int
        Dimension of latent factors.
    n_epochs : int
        Number of training epochs.
    lr : float
        Learning rate (eta).
    reg : float
        L2 regularization strength (lambda).
    clip_min, clip_max : float
        Clipping range for predictions.
    random_state : int
    """

    def __init__(self, n_factors=20, n_epochs=20, lr=0.005, reg=0.02,
                 clip_min=0.5, clip_max=5.0, random_state=42):
        self.n_factors  = n_factors
        self.n_epochs   = n_epochs
        self.lr         = lr
        self.reg        = reg
        self.clip_min   = clip_min
        self.clip_max   = clip_max
        self.rng        = np.random.default_rng(random_state)

    # ── Private helpers ────────────────────────────────────────────────────
    def _init_params(self, n_users, n_items):
        k = self.n_factors
        scale = 0.1
        self.bu_ = np.zeros(n_users)
        self.bi_ = np.zeros(n_items)
        self.P_  = self.rng.normal(0, scale, (n_users, k))   # user factors
        self.Q_  = self.rng.normal(0, scale, (n_items, k))   # item factors

    def _predict_one(self, u_idx, i_idx):
        p = self.mu_ + self.bu_[u_idx] + self.bi_[i_idx] + self.P_[u_idx] @ self.Q_[i_idx]
        return np.clip(p, self.clip_min, self.clip_max)

    # ── Public API ─────────────────────────────────────────────────────────
    def fit(self, train_df, val_df=None, target_col="userPredictRating",
            user_col="userId", item_col="movieId"):
        """Fit Biased MF using SGD. Tracks train/val RMSE per epoch."""
        # Build index maps
        all_users = pd.concat([train_df[user_col], val_df[user_col] if val_df is not None else train_df[user_col]]).unique()
        all_items = pd.concat([train_df[item_col], val_df[item_col] if val_df is not None else train_df[item_col]]).unique()
        self.user2idx_ = {u: i for i, u in enumerate(all_users)}
        self.item2idx_ = {m: i for i, m in enumerate(all_items)}
        n_users = len(all_users)
        n_items = len(all_items)

        self.mu_ = train_df[target_col].mean()
        self._init_params(n_users, n_items)

        # Convert to arrays for speed
        u_arr = train_df[user_col].map(self.user2idx_).values
        i_arr = train_df[item_col].map(self.item2idx_).values
        r_arr = train_df[target_col].values.astype(float)

        self.train_rmse_history_ = []
        self.val_rmse_history_   = []

        for epoch in tqdm(range(self.n_epochs), desc="Training BiasedMF"):
            # Shuffle
            idx = self.rng.permutation(len(r_arr))
            u_arr_s, i_arr_s, r_arr_s = u_arr[idx], i_arr[idx], r_arr[idx]

            for u_idx, i_idx, r in zip(u_arr_s, i_arr_s, r_arr_s):
                pred = self.mu_ + self.bu_[u_idx] + self.bi_[i_idx] + self.P_[u_idx] @ self.Q_[i_idx]
                err  = r - pred

                # Update biases
                self.bu_[u_idx] += self.lr * (err - self.reg * self.bu_[u_idx])
                self.bi_[i_idx] += self.lr * (err - self.reg * self.bi_[i_idx])

                # Update latent factors
                pu_old = self.P_[u_idx].copy()
                self.P_[u_idx] += self.lr * (err * self.Q_[i_idx] - self.reg * self.P_[u_idx])
                self.Q_[i_idx] += self.lr * (err * pu_old           - self.reg * self.Q_[i_idx])

            # Record RMSE
            train_preds = self._predict_batch(train_df, user_col, item_col)
            self.train_rmse_history_.append(rmse(r_arr, train_preds))

            if val_df is not None:
                y_val_ep = val_df[target_col].values
                val_preds = self._predict_batch(val_df, user_col, item_col)
                self.val_rmse_history_.append(rmse(y_val_ep, clip_preds(val_preds)))

        return self

    def _predict_batch(self, df, user_col="userId", item_col="movieId"):
        u_idxs = df[user_col].map(self.user2idx_).fillna(-1).astype(int).values
        i_idxs = df[item_col].map(self.item2idx_).fillna(-1).astype(int).values
        preds = np.full(len(df), self.mu_)
        for j, (u, i) in enumerate(zip(u_idxs, i_idxs)):
            if u == -1 and i == -1:
                preds[j] = self.mu_
            elif u == -1:
                preds[j] = self.mu_ + (self.bi_[i] if 0 <= i < len(self.bi_) else 0)
            elif i == -1:
                preds[j] = self.mu_ + (self.bu_[u] if 0 <= u < len(self.bu_) else 0)
            else:
                preds[j] = self.mu_ + self.bu_[u] + self.bi_[i] + self.P_[u] @ self.Q_[i]
        return np.clip(preds, self.clip_min, self.clip_max)

    def predict(self, df, user_col="userId", item_col="movieId"):
        """Return predictions (clipped to [clip_min, clip_max])."""
        return self._predict_batch(df, user_col, item_col)


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping BiasedMF training.")
else:
    bmf = BiasedMF(n_factors=20, n_epochs=25, lr=0.005, reg=0.02, random_state=RANDOM_SEED)
    bmf.fit(train_df, val_df=val_df, target_col="userPredictRating")
    print("Training complete.")


In [ ]:
if DATA_AVAILABLE:
    # ── Learning Curve ────────────────────────────────────────────────────
    plt.figure(figsize=(9, 4))
    epochs = range(1, len(bmf.train_rmse_history_) + 1)
    plt.plot(epochs, bmf.train_rmse_history_, label="Train RMSE", marker="o", markersize=3)
    if bmf.val_rmse_history_:
        plt.plot(epochs, bmf.val_rmse_history_, label="Val RMSE", marker="s", markersize=3, linestyle="--")
    plt.xlabel("Epoch")
    plt.ylabel("RMSE")
    plt.title("BiasedMF – Learning Curve", fontsize=13)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
if DATA_AVAILABLE:
    bmf_val_preds  = bmf.predict(val_df)
    bmf_test_preds = bmf.predict(test_df)

    res = evaluate("BiasedMF (val)",  y_val,  bmf_val_preds)
    results.append({**res, "split": "val"})
    res = evaluate("BiasedMF (test)", y_test, bmf_test_preds)
    results.append({**res, "split": "test"})


---
## Section 5 – Result Analysis & Visualizations

- Bảng so sánh RMSE / MAE tất cả models  
- Biểu đồ bar so sánh  
- Phân tích lỗi theo `userCertainty`  
- Scatter plot: actual vs predicted (BiasedMF)


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping result analysis section.")
else:
    results_df = pd.DataFrame(results)
    test_results = results_df[results_df["split"] == "test"].copy().reset_index(drop=True)
    print("=== Test Set Performance ===")
    display(test_results[["model", "RMSE", "MAE"]].sort_values("RMSE"))


In [ ]:
if DATA_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    test_sorted = test_results.sort_values("RMSE")

    for ax, metric in zip(axes, ["RMSE", "MAE"]):
        bars = ax.barh(test_sorted["model"], test_sorted[metric],
                       color=sns.color_palette("viridis", len(test_sorted)))
        ax.set_xlabel(metric)
        ax.set_title(f"Test {metric} – All Models", fontsize=13)
        ax.invert_yaxis()
        for bar, val in zip(bars, test_sorted[metric]):
            ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                    f"{val:.4f}", va="center", fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
if DATA_AVAILABLE:
    # ── Error analysis by userCertainty ────────────────────────────────────
    test_analysis = test_df.copy()
    test_analysis["bmf_pred"]  = bmf_test_preds
    test_analysis["abs_error"] = (test_analysis["userPredictRating"] - test_analysis["bmf_pred"]).abs()

    if "userCertainty" in test_analysis.columns:
        cert_error = (
            test_analysis.groupby("userCertainty")["abs_error"]
            .agg(["mean", "std", "count"])
            .reset_index()
        )
        print("=== MAE by userCertainty (BiasedMF) ===")
        display(cert_error)

        plt.figure(figsize=(8, 4))
        plt.bar(cert_error["userCertainty"].astype(str), cert_error["mean"],
                yerr=cert_error["std"] / np.sqrt(cert_error["count"]),
                capsize=4, color=sns.color_palette("muted", len(cert_error)))
        plt.xlabel("userCertainty (1=very uncertain, 5=very certain)")
        plt.ylabel("Mean Absolute Error")
        plt.title("BiasedMF – Prediction Error by User Certainty", fontsize=13)
        plt.tight_layout()
        plt.show()
    else:
        print("[INFO] 'userCertainty' column not found – skipping certainty analysis.")


In [ ]:
if DATA_AVAILABLE:
    # ── Scatter: actual vs predicted ────────────────────────────────────────
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, bmf_test_preds, alpha=0.15, s=8, color=sns.color_palette("muted")[0])
    lo, hi = CLIP_MIN - 0.1, CLIP_MAX + 0.1
    plt.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Perfect prediction")
    plt.xlim(lo, hi); plt.ylim(lo, hi)
    plt.xlabel("Actual userPredictRating"); plt.ylabel("Predicted")
    plt.title("Actual vs Predicted – BiasedMF (Test Set)", fontsize=13)
    plt.legend(); plt.tight_layout(); plt.show()


---
## Section 6 – Ablation / Leakage Demo

### 6.1 Ý nghĩa của leakage

`systemPredictRating` trong `belief_data.csv` là output của recommender nội bộ.  
Nếu dùng làm feature:
- Model có thể chỉ học lại "phiên bản đơn giản" của hệ thống đó.
- Metric trở nên quá lạc quan và không phản ánh đúng khả năng của model.
- Trong triển khai thực tế, bạn sẽ không có `systemPredictRating` cho user/item mới.

### 6.2 Thí nghiệm ablation: Ridge với và không có `systemPredictRating`

So sánh:
- **Ridge (no leak)**: chỉ dùng genre + user/movie stats  
- **Ridge (with system score)**: thêm `systemPredictRating` làm feature


In [ ]:
if not DATA_AVAILABLE:
    print("[SKIP] Data not available – skipping ablation section.")
else:
    if "systemPredictRating" in train_df.columns:
        # Version WITH system score (leaky)
        LEAK_COLS = FEATURE_COLS + ["systemPredictRating"]
        X_tr_leak = train_df[LEAK_COLS].fillna(0).values
        X_v_leak  = val_df[LEAK_COLS].fillna(0).values
        X_ts_leak = test_df[LEAK_COLS].fillna(0).values

        ridge_leak = Ridge(alpha=1.0, random_state=RANDOM_SEED)
        ridge_leak.fit(X_tr_leak, y_train)

        res_nl = evaluate("Ridge – no leakage (test)", y_test, ridge.predict(X_test))
        res_lk = evaluate("Ridge – WITH systemPredictRating [LEAKY] (test)",
                           y_test, ridge_leak.predict(X_ts_leak))

        leakage_results = pd.DataFrame([res_nl, res_lk])
        print("\n=== Ablation: Leakage Effect ===")
        display(leakage_results[["model", "RMSE", "MAE"]])

        # Visualize
        fig, ax = plt.subplots(figsize=(8, 3))
        x = np.arange(2)
        bars = ax.bar(["Ridge\n(no leakage)", "Ridge\n(+systemPredictRating)"],
                      leakage_results["RMSE"].values,
                      color=[sns.color_palette("muted")[2], sns.color_palette("muted")[3]],
                      width=0.4)
        ax.set_ylabel("RMSE")
        ax.set_title("Ablation: Impact of systemPredictRating (Leakage Demo)", fontsize=13)
        for bar, val in zip(bars, leakage_results["RMSE"].values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f"{val:.4f}", ha="center", fontsize=10)
        plt.tight_layout(); plt.show()

        print("\n💡 Conclusion: The 'leaky' model achieves lower RMSE because it reuses")
        print("   the system's own predictions. This is NOT a genuine improvement.")
        print("   The main model (BiasedMF) does NOT use systemPredictRating.")
    else:
        print("[INFO] 'systemPredictRating' column not found in data – ablation skipped.")
        print("       The no-leakage principle still applies to the main model.")


---
## Section 7 – Conclusion

### 7.1 Trả lời Research Question

> *Liệu lịch sử đánh giá của người dùng và thông tin thể loại phim có đủ để dự đoán tốt belief rating cho phim chưa xem? Và độ chắc chắn (userCertainty) có tương quan với sai số dự đoán không?*

**Kết quả tóm tắt:**

| Model | RMSE (test) | Nhận xét |
|-------|-------------|----------|
| Global Mean | *(xem kết quả)* | Baseline đơn giản nhất |
| User Mean | *(xem kết quả)* | Cải thiện nhờ personalisation |
| Movie Mean | *(xem kết quả)* | Khác biệt theo phim |
| Additive Bias | *(xem kết quả)* | Kết hợp cả hai |
| Ridge Regression | *(xem kết quả)* | Thêm genre + stats |
| Random Forest | *(xem kết quả)* | Non-linear baseline |
| **BiasedMF (SGD)** | *(xem kết quả)* | **Model chính** |

Biased MF nắm bắt được tương tác user-item latent ngoài bias đơn giản, thường cho RMSE thấp hơn các baseline tuyến tính. Kết quả cụ thể phụ thuộc dữ liệu thực tế.

Phân tích `userCertainty`: người dùng tự báo cáo certainty cao hơn (4–5) thường có sai số *thấp hơn* (model dự đoán tốt hơn), điều này hợp lý vì user chắc chắn hơn thường có signal lịch sử rõ ràng hơn.

---

### 7.2 Limitations

1. **Selection bias**: chỉ có dữ liệu belief của những user đã tham gia elicitation. Không đại diện toàn bộ user MovieLens.

2. **MNAR (Missing Not At Random)**: người dùng chỉ trả lời khi họ biết / quan tâm đến phim → belief bị thiên lệch về phía phim "nổi tiếng".

3. **Leakage từ `systemPredictRating`**: model chính không dùng; nhưng đây là ví dụ quan trọng về cách leakage làm méo metric.

4. **Cold-start**: user/movie mới chưa có lịch sử → fallback về global mean / bias không cá nhân hoá được.

5. **Content features hạn chế**: chỉ có genre; không có metadata ngoài (poster, plot, diễn viên…).

6. **Temporal drift**: sở thích user thay đổi theo thời gian; model tĩnh không nắm bắt được drift dài hạn.

---

### 7.3 Next Steps

- **Temporal MF / TimeSVD++**: mô hình hoá drift theo thời gian.
- **Uncertainty-aware model**: dự đoán phân phối thay vì điểm, dùng `userCertainty` làm supervision signal.
- **Content-based features**: thêm metadata từ TMDB (plot, cast, director).
- **Exposure bias correction**: dùng `user_recommendation_history.csv` để tính inverse propensity weights.
- **Neural CF (NCF)**: tăng biểu đạt nhờ non-linear interactions.


In [ ]:
# Final summary of all test-set results
if DATA_AVAILABLE:
    final = results_df[results_df["split"] == "test"].sort_values("RMSE")
    print("=" * 60)
    print("FINAL SUMMARY – Test Set")
    print("=" * 60)
    display(final[["model", "RMSE", "MAE"]].reset_index(drop=True))
